# Generate Enhanced Queries: Aya Expanse 8B

**Model:** CohereForAI/aya-expanse-8b

**Hardware:** A100 GPU (40GB VRAM)

**Quantization:** 4-bit (required for 8B model)

**Temperature:** 0.1 (more focused)

**Batch Size:** 8 (smaller due to 8B size)

**Note:** This is a GATED model - you must accept the license and login to HuggingFace

**License:** CC-BY-NC-4.0 (non-commercial use only, fine for thesis)

---

## Setup

### Step 1: Clone Repository and Install Dependencies

In [1]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets accelerate bitsandbytes huggingface_hub

print("\n" + "="*60)
print("Installation complete")
print("="*60)
print("IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' -> 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

Cloning into 'graduation'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 552 (delta 31), reused 73 (delta 23), pack-reused 468 (from 1)
Receiving objects: 100% (552/552), 20.54 MiB | 8.24 MiB/s, done.
Resolving deltas: 100% (203/203), done.
/content/graduation/arabic-rag-query-enhancement
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/j

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\nEnvironment configured")
print("Ready to run experiment")

Mounted at /content/drive
/content/graduation/arabic-rag-query-enhancement
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

Environment configured
Ready to run experiment


### Step 3: HuggingFace Login (REQUIRED for Aya Expanse)

Aya Expanse is a gated model. You must:
1. Accept the license at: https://huggingface.co/CohereForAI/aya-expanse-8b
2. Get your HuggingFace token from: https://huggingface.co/settings/tokens
3. Run the cell below and paste your token when prompted

In [2]:
from huggingface_hub import login

# Login to HuggingFace (you'll be prompted for token)
login()

print("\nLogged in to HuggingFace")
print("\nIMPORTANT: Make sure you've accepted the Aya Expanse license at:")
print("https://huggingface.co/CohereForAI/aya-expanse-8b")
print("\nIf you haven't accepted it, the model download will fail.")


Logged in to HuggingFace

IMPORTANT: Make sure you've accepted the Aya Expanse license at:
https://huggingface.co/CohereForAI/aya-expanse-8b

If you haven't accepted it, the model download will fail.


## Import Modules

In [3]:
from src.utils.data_loader import MIRACLDataLoader

import torch
from tqdm.notebook import tqdm

print("Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Modules imported
GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.5 GB


## Load Data

In [4]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels: 2896

Sample Query:
  ID: 8099
  Text: من هو علي بن محمد السمري؟
  Relevant docs: 10


## Load Aya Expanse 8B with 4-bit Quantization

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Loading Aya Expanse 8B with 4-bit quantization...")
print("Model: CohereForAI/aya-expanse-8b")
print("Quantization: 4-bit NF4")
print("Temperature: 0.1")
print("Batch size: 8")
print("\nThis will download ~5GB (quantized)...\n")

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("CohereForAI/aya-expanse-8b")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    "CohereForAI/aya-expanse-8b",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

print(f"\nAya Expanse 8B loaded")
print(f"Model device: {model.device}")
print(f"Quantization: 4-bit NF4")

Loading Aya Expanse 8B with 4-bit quantization...
Model: CohereForAI/aya-expanse-8b
Quantization: 4-bit NF4
Temperature: 0.1
Batch size: 8

This will download ~5GB (quantized)...



config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


Aya Expanse 8B loaded
Model device: cuda:0
Quantization: 4-bit NF4


## Create Aya Expanse Enhancer

In [9]:
class AyaExpanseEnhancer:
    def __init__(self, model, tokenizer, max_new_tokens=128, temperature=0.1, batch_size=8):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = 0.9
        self.batch_size = batch_size
        self.system_prompt = (
            "You are asked to write a passage that answers the given query. "
            "Do not ask the user for further clarification. "
            "Respond in Arabic only."
        )

    def enhance(self, query, query_id=None):
        """Enhance single query using Aya Expanse"""
        # Format with Aya's chat template
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": query}
        ]

        # Apply chat template and get text first
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Then tokenize
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                top_p=self.top_p,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        # Decode only generated part
        generated = outputs[0][inputs.input_ids.shape[1]:]
        pseudo_doc = self.tokenizer.decode(generated, skip_special_tokens=True)

        return f"{query} {pseudo_doc}"

    def enhance_batch_parallel(self, queries, query_ids=None):
        """Enhance batch of queries"""
        # Format all messages
        all_messages = [
            [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": query}
            ]
            for query in queries
        ]

        # Apply chat template to get text first
        texts = [
            self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            for messages in all_messages
        ]

        # Then tokenize all with padding
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(self.model.device)

        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                top_p=self.top_p,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        # Decode all
        input_length = inputs.input_ids.shape[1]
        pseudo_docs = self.tokenizer.batch_decode(
            outputs[:, input_length:],
            skip_special_tokens=True
        )

        return [f"{q} {doc}" for q, doc in zip(queries, pseudo_docs)]

    def enhance_batch(self, queries, query_ids=None, show_progress=True):
        """Enhance with batching"""
        enhanced = []
        num_batches = (len(queries) + self.batch_size - 1) // self.batch_size

        iterator = tqdm(range(num_batches), desc="Enhancing batches") if show_progress else range(num_batches)

        for batch_idx in iterator:
            start_idx = batch_idx * self.batch_size
            end_idx = min(start_idx + self.batch_size, len(queries))
            batch_queries = queries[start_idx:end_idx]
            batch_enhanced = self.enhance_batch_parallel(batch_queries)
            enhanced.extend(batch_enhanced)

        return enhanced

# Recreate enhancer with fix
enhancer = AyaExpanseEnhancer(model, tokenizer, max_new_tokens=128, temperature=0.1, batch_size=32)

print("✓ Aya Expanse enhancer recreated with fix")
print("✓ Now using two-step process: chat_template -> text -> tokenize")


✓ Aya Expanse enhancer recreated with fix
✓ Now using two-step process: chat_template -> text -> tokenize


## Check GPU Memory

In [6]:
if torch.cuda.is_available():
    print("=== GPU Status ===")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"Memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.2f} GB")

=== GPU Status ===
GPU: NVIDIA A100-SXM4-40GB
Memory allocated: 5.31 GB
Memory reserved: 14.93 GB
Memory total: 39.49 GB
Memory free: 34.19 GB


## Test on Sample Query

In [10]:
# Test enhancer on sample query
sample_query = topics[sample_qid]['title']
print(f"Testing enhancer on sample query...\n")
print(f"Original: {sample_query}")
print(f"\nGenerating pseudo-document...")

enhanced_sample = enhancer.enhance(sample_query)
print(f"\nEnhanced: {enhanced_sample[:500]}...")  # Show first 500 chars
print(f"\nLength: {len(sample_query)} -> {len(enhanced_sample)} chars")
print(f"Expansion ratio: {len(enhanced_sample)/len(sample_query):.2f}x")

Testing enhancer on sample query...

Original: من هو علي بن محمد السمري؟

Generating pseudo-document...

Enhanced: من هو علي بن محمد السمري؟ علي بن محمد السمري هو عالم دين شيعي وإمام ومفسر إسلامي معروف. ولد في إيران واشتهر بتفسيره للقرآن الكريم وتأليفه لكتب في الفقه والتفسير. يتمتع السمري بشهرة واسعة في الأوساط الشيعية، خاصة بين أتباع مدرسة أهل البيت (عليهم السلام).

يُعرف السمري بأسلوبه الفريد في تفسير القرآن، حيث يجمع بين التفسير اللغوي والروحي، ويُشدد على أهمية فهم الن...

Length: 25 -> 361 chars
Expansion ratio: 14.44x


## Generate Enhanced Queries for All Data

In [11]:
import time

print("="*60)
print("GENERATING ENHANCED QUERIES: Aya Expanse 8B")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nTotal queries: {len(query_texts)}")
print(f"Batch size: 8")
print(f"Expected batches: {len(query_texts) // 8 + 1}")
print(f"Expected time: ~40-50 minutes (slower due to 4-bit quantization)\n")

start_time = time.time()

# Apply Query2Doc enhancement
enhanced_queries = enhancer.enhance_batch(
    query_texts,
    query_ids,
    show_progress=True
)

elapsed = time.time() - start_time

print(f"\nEnhanced {len(enhanced_queries)} queries")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Queries per minute: {len(query_texts)/(elapsed/60):.1f}")

GENERATING ENHANCED QUERIES: Aya Expanse 8B

Total queries: 2896
Batch size: 8
Expected batches: 363
Expected time: ~40-50 minutes (slower due to 4-bit quantization)



Enhancing batches:   0%|          | 0/91 [00:00<?, ?it/s]


Enhanced 2896 queries
Total time: 21.4 minutes
Queries per minute: 135.2


## Show Enhancement Examples

In [12]:
print("\nEnhancement Examples:\n")
for i in range(min(5, len(query_texts))):
    print(f"Query {i+1}:")
    print(f"  Original ({len(query_texts[i])} chars): {query_texts[i]}")
    print(f"  Enhanced ({len(enhanced_queries[i])} chars): {enhanced_queries[i][:200]}...")  # First 200 chars
    print(f"  Expansion: {len(enhanced_queries[i])/len(query_texts[i]):.2f}x")
    print()


Enhancement Examples:

Query 1:
  Original (25 chars): من هو علي بن محمد السمري؟
  Enhanced (384 chars): من هو علي بن محمد السمري؟ علي بن محمد السمري هو عالم دين شيعي وإمام ومفسر إسلامي معروف. ولد في العراق واشتهر بتفسيره للقرآن الكريم وتأليفه لكتب في الفقه والتفسير. يُعتبر السمري من الشخصيات المؤثرة في ...
  Expansion: 15.36x

Query 2:
  Original (34 chars): متى تم إستخدام الغوّاصات لأول مرة؟
  Enhanced (401 chars): متى تم إستخدام الغوّاصات لأول مرة؟ تم استخدام الغوّاصات لأول مرة في القرن التاسع عشر، وتحديدًا في منتصف القرن 1880. شهدت هذه الفترة ظهور أولى الغوّاصات العملية التي تم تصميمها للتنقل تحت الماء.

أحد ا...
  Expansion: 11.79x

Query 3:
  Original (28 chars): من هو القديس المسمى بالصخرة؟
  Enhanced (365 chars): من هو القديس المسمى بالصخرة؟ القديس يوحنا المعمدان، المعروف أيضًا باسم "القديس يوحنا المعمدان" أو "القديس يوحنا البشير"، هو شخصية دينية مهمة في المسيحية. يُعتبر يوحنا المعمدان أحد الأنبياء في العهد ال...
  Expansion: 13.04x

Query 4:
  Original (33 chars): هل يرتبط ال

## Query Expansion Statistics

In [13]:
import numpy as np

# Calculate statistics
original_lengths = [len(q) for q in query_texts]
enhanced_lengths = [len(eq) for eq in enhanced_queries]
expansion_ratios = [e/o if o > 0 else 0 for o, e in zip(original_lengths, enhanced_lengths)]

print("=== Query Expansion Statistics ===")
print(f"\nOriginal queries:")
print(f"  Mean length: {np.mean(original_lengths):.1f} chars")
print(f"  Median length: {np.median(original_lengths):.1f} chars")
print(f"  Min/Max: {min(original_lengths)} / {max(original_lengths)} chars")

print(f"\nEnhanced queries:")
print(f"  Mean length: {np.mean(enhanced_lengths):.1f} chars")
print(f"  Median length: {np.median(enhanced_lengths):.1f} chars")
print(f"  Min/Max: {min(enhanced_lengths)} / {max(enhanced_lengths)} chars")

print(f"\nExpansion ratio:")
print(f"  Mean: {np.mean(expansion_ratios):.2f}x")
print(f"  Median: {np.median(expansion_ratios):.2f}x")
print(f"  Min/Max: {min(expansion_ratios):.2f}x / {max(expansion_ratios):.2f}x")

=== Query Expansion Statistics ===

Original queries:
  Mean length: 29.5 chars
  Median length: 27.0 chars
  Min/Max: 12 / 101 chars

Enhanced queries:
  Mean length: 352.9 chars
  Median length: 394.0 chars
  Min/Max: 29 / 557 chars

Expansion ratio:
  Mean: 13.60x
  Median: 13.67x
  Min/Max: 1.18x / 34.58x


## Save Enhanced Queries

In [14]:
import pickle

# Save enhanced queries
output_file = 'enhanced_queries_aya_expanse_8b.pkl'

with open(output_file, 'wb') as f:
    pickle.dump({
        'query_ids': query_ids,
        'original': query_texts,
        'enhanced': enhanced_queries,
        'model': 'CohereForAI/aya-expanse-8b',
        'config': {
            'max_new_tokens': 128,
            'temperature': 0.1,
            'top_p': 0.9,
            'batch_size': 8,
            'quantization': '4-bit NF4',
            'hardware': 'A100 GPU'
        },
        'stats': {
            'total_queries': len(query_texts),
            'mean_original_length': np.mean(original_lengths),
            'mean_enhanced_length': np.mean(enhanced_lengths),
            'mean_expansion_ratio': np.mean(expansion_ratios),
            'generation_time_minutes': elapsed/60
        }
    }, f)

print(f"Enhanced queries saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024**2:.1f} MB")

# Also save to Google Drive
drive_path = '/content/drive/MyDrive/enhanced_queries_aya_expanse_8b.pkl'
!cp {output_file} {drive_path}
print(f"\nBackup saved to Google Drive: {drive_path}")

Enhanced queries saved to: enhanced_queries_aya_expanse_8b.pkl
File size: 1.9 MB

Backup saved to Google Drive: /content/drive/MyDrive/enhanced_queries_aya_expanse_8b.pkl


## Summary

In [15]:
print("="*60)
print("GENERATION COMPLETE")
print("="*60)
print(f"\nModel: Aya Expanse 8B")
print(f"Quantization: 4-bit NF4")
print(f"Temperature: 0.1")
print(f"Batch size: 8")
print(f"Hardware: {torch.cuda.get_device_name(0)}")
print(f"\nQueries processed: {len(enhanced_queries)}")
print(f"Generation time: {elapsed/60:.1f} minutes")
print(f"Average expansion: {np.mean(expansion_ratios):.2f}x")
print(f"\nOutput file: {output_file}")
print(f"\nNote: Aya Expanse is purpose-built for multilingual tasks (101 languages).")
print(f"License: CC-BY-NC-4.0 (non-commercial use only)")
print(f"\nNext step: Use evaluate_enhanced_queries.ipynb to test with Dense and BM25")

GENERATION COMPLETE

Model: Aya Expanse 8B
Quantization: 4-bit NF4
Temperature: 0.1
Batch size: 8
Hardware: NVIDIA A100-SXM4-40GB

Queries processed: 2896
Generation time: 21.4 minutes
Average expansion: 13.60x

Output file: enhanced_queries_aya_expanse_8b.pkl

Note: Aya Expanse is purpose-built for multilingual tasks (101 languages).
License: CC-BY-NC-4.0 (non-commercial use only)

Next step: Use evaluate_enhanced_queries.ipynb to test with Dense and BM25
